In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parents[0]))
from market_inventory.universe import CoinUniverse, ProjectUniverse
from market_inventory.inventory import inventory_crypto_markets
from market_inventory.polymarket_clients import GammaClient
import pandas as pd
pd.set_option('display.max_colwidth', None)
import logging

logging.basicConfig(
    level=logging.INFO,   # set DEBUG for even more detail
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s"
)

coin_universe = CoinUniverse.from_json(Path("/workspaces/Agents_v1/coins_universe.json"))
project_universe = ProjectUniverse.from_json(Path("/workspaces/Agents_v1/projects_universe.json"))
gamma = GammaClient()

df = inventory_crypto_markets(
    gamma=gamma,
    coin_universe=coin_universe,
    project_universe=project_universe,
    limit_events=500,
)

2026-02-18 06:57:15,608 | INFO | httpx | HTTP Request: GET https://gamma-api.polymarket.com/tags/slug/crypto "HTTP/1.1 200 OK"
2026-02-18 06:57:15,662 | INFO | httpx | HTTP Request: GET https://gamma-api.polymarket.com/events?tag_id=21&active=true&closed=false&limit=100&offset=0 "HTTP/1.1 200 OK"
2026-02-18 06:57:16,315 | INFO | httpx | HTTP Request: GET https://gamma-api.polymarket.com/events?tag_id=21&active=true&closed=false&limit=100&offset=100 "HTTP/1.1 200 OK"
2026-02-18 06:57:16,476 | INFO | httpx | HTTP Request: GET https://gamma-api.polymarket.com/events?tag_id=21&active=true&closed=false&limit=100&offset=200 "HTTP/1.1 200 OK"


In [4]:
from market_inventory.inventory import get_crypto_tag_id


tag_id = get_crypto_tag_id(gamma)
print(tag_id)

2026-02-18 07:04:18,923 | INFO | httpx | HTTP Request: GET https://gamma-api.polymarket.com/tags/slug/crypto "HTTP/1.1 200 OK"


21


In [6]:
gamma = GammaClient()
gamma.list_events(tag_id=21,active=True,closed=True,limit=5)

2026-02-18 07:05:24,231 | INFO | httpx | HTTP Request: GET https://gamma-api.polymarket.com/events?tag_id=21&active=true&closed=true&limit=5&offset=0 "HTTP/1.1 200 OK"


[{'id': '10035',
  'ticker': 'will-eth-or-sol-reach-alltime-high-first',
  'slug': 'will-eth-or-sol-reach-alltime-high-first',
  'title': 'Will ETH or SOL reach all-time high first?',
  'description': 'This market will resolve to “ETH” if Ethereum reaches an all-time high before Solana. This market will resolve to “SOL” if Solana reaches an all-time high before Ethereum.\n\nThe resolution source will be Binance, specifically the candlestick high prices for both ETH_USDT (https://www.binance.com/en/trade/ETH_USDT) and SOL_USDT (https://www.binance.com/en/trade/SOL_USDT). The current all-time candlestick highs used for this market for ETH and SOL are $4,868.00 and $259.90 respectively.\n\nIf neither ETH nor SOL reach an all time high by December 31, 2024, 11:59 PM ET, this market will resolve 50-50.',
  'resolutionSource': '',
  'startDate': '2024-03-16T01:19:28.137088Z',
  'creationDate': '2024-03-14T19:24:14.480304Z',
  'endDate': '2024-12-31T12:00:00Z',
  'image': 'https://polymarket-

In [ ]:
from pipeline.historical_triage_runner import triage_dataframe_async

df_out = await triage_dataframe_async(
    df,
    max_rows=1,
    max_concurrency=1,
    row_timeout_s=120.0,
    total_timeout_s=300.0,
    log_every=1,)


## Connector Builder Agent Test

Tests the `connector_builder_agent` end-to-end:

1. **Imports** – load all connector-builder symbols
2. **DataSourcePlan** – construct a sample plan (CoinGecko BTC price, free API)
3. **`build_connector`** – call the agent for that single plan; inspect metadata
4. **Source code** – display the generated function + imports
5. **`build_connectors_async`** – exercise the batch runner via a mini triaged DataFrame
6. **`write_connectors_module`** – write the generated connectors to a `.py` file and display it


In [ ]:

# ── 2. Imports for connector builder test ────────────────────────────────────
import json
import tempfile
from pathlib import Path

import pandas as pd

from parsing.connector_types import ConnectorType
from parsing.historical_data_triage_models import DataSourcePlan
from parsing.connector_models import ConnectorCode
from pm_agents.connector_builder_agent import build_connector
from pipeline.connector_builder_runner import build_connectors_async, write_connectors_module


In [ ]:

# ── 3. Build a sample DataSourcePlan ─────────────────────────────────────────
# A concrete free-API plan targeting CoinGecko's market_chart endpoint.
# This is the input the connector_builder_agent will use to generate a Python connector.
plan = DataSourcePlan(
    connector_type=ConnectorType.FREE_API_GENERIC,
    connector_key="free_api_generic:api.coingecko.com/api/v3/coins/bitcoin/market_chart",
    series_id="btc_price_usd_daily",
    method="api",
    target="CoinGecko public API – BTC/USD daily price history",
    url_or_endpoint_hint="https://api.coingecko.com/api/v3/coins/bitcoin/market_chart",
    access="rate_limited_free",
    paywall_evidence=None,
    effort="low",
    reliability="high",
    notes=None,
    extraction_target="Daily closing price of Bitcoin in USD.",
    extraction_method_detail=(
        "GET https://api.coingecko.com/api/v3/coins/bitcoin/market_chart"
        "?vs_currency=usd&days=365&interval=daily. "
        "Response JSON has key 'prices': a list of [unix_timestamp_ms, price] pairs. "
        "Convert each unix_timestamp_ms (divide by 1000) to a datetime.date via "
        "datetime.utcfromtimestamp(). Return one row per day sorted chronologically."
    ),
    value_parse_pattern="Plain float from the JSON response – no transformation needed.",
    page_interaction_required=None,
    rendering_notes="Server-side JSON API – no JS rendering required.",
    output_columns=["date", "price_usd"],
    connector_function_name="fetch_coingecko_btc_price_usd",
    required_params={"vs_currency": "usd", "days": 365, "interval": "daily"},
)

print(json.dumps(plan.model_dump(mode="json"), indent=2))


In [ ]:

# ── 4. Call build_connector and inspect metadata ──────────────────────────────
code = await build_connector(plan, timeout_s=120.0)

print(f"connector_key          : {code.connector_key}")
print(f"connector_function_name: {code.connector_function_name}")
print(f"series_id              : {code.series_id}")
print(f"connector_type         : {code.connector_type}")
print(f"output_columns         : {code.output_columns}")
print(f"dependencies           : {code.dependencies}")
print(f"notes                  : {code.notes}")


In [ ]:

# ── 5. Display the generated source code ─────────────────────────────────────
print("=== imports ===")
for line in code.imports:
    print(line)

print("\n=== source_code ===")
print(code.source_code)


In [ ]:

# ── 6. Test build_connectors_async via a mini triaged DataFrame ───────────────
# Construct a DataFrame as if it came from triage_dataframe_async.
# The only required column is triage_plans_json (a JSON-serialised list of DataSourcePlan dicts).
mini_df = pd.DataFrame([{
    "question": "Will BTC exceed $100k by end of 2025?",
    "triage_plans_json": json.dumps([plan.model_dump(mode="json")]),
}])

registry = await build_connectors_async(
    mini_df,
    max_concurrency=1,
    max_plans=1,
    plan_timeout_s=120.0,
    total_timeout_s=180.0,
)

print(f"Registry keys: {list(registry.keys())}")
for key, c in registry.items():
    print(f"\n  {key}")
    print(f"  fn   : {c.connector_function_name}")
    print(f"  deps : {c.dependencies}")
    print(f"  notes: {c.notes}")


In [ ]:

# ── 7. Write connector module to a temp file and display it ──────────────────
import tempfile

with tempfile.NamedTemporaryFile(suffix=".py", delete=False, mode="w") as f:
    tmp_path = Path(f.name)

write_connectors_module(registry, tmp_path)
print(tmp_path.read_text())
